---
# SIT744 Assignment 1 (T2 2025)

Due: Week 3 Friday 8:00 pm (AEST)

---

This is an individual assignment. It contributes 20% to your final mark. Read the assignment instructions carefully.

## What to submit
By the due date, you are required to submit the following files to the corresponding Assignment (Dropbox) in CloudDeakin:

- **[YourID]_[UnitCode]_assignment1_solution.ipynb**: This is your Python notebook solution source file.
- **[YourID]_[UnitCode]_assingment1_output.pdf**: This is the output of your Python notebook solution exported in PDF format. (You may use [nbconvert](https://github.com/jupyter/nbconvert).)
- (HD Task only) A short (less than 5 minutes) screencast explaining your work (including study design, implementation, and main conclusions).
- Extra files required to complete your assignment, if any (e.g., images used in your answers).

For example, if your student ID is: 123456, and you are a SIT744 student, you will then need to submit the following files:

- 123456_SIT744_assignment1_solution.ipynb
- 123456_SIT744_assignment1_output.pdf

Please keep your answers short and to the point. Clean up your code outputs to reduce unnecessary information (e.g., excessively long training logs).


## Assignment objective

This assignment is for you to demonstrate the knowledge in deep learning that you have acquired from the lectures and practical lab materials in weeks 1-3. Going through these materials before/while attempting this assignment is highly recommended.


This assignment consists of four sets of tasks with progressive level of challenges.

- Set 1 (P Tasks)
- Set 2 (C Tasks)
- Set 3 (D Tasks)
- Set 4 (HD Tasks)

Set 1 is labeled as P tasks because these demonstrate the minimum requirements of the unit. Concentrate on the P tasks first as these are the foundation for all the rest. As you move through sets 2, 3 and 4, the tasks become more challenging, with each allowing you to obtain the marks required to achieve the next achievement level (C, D, and HD respectively).

## Marking criteria
Indicative weights of various tasks are provided below, but your submission will be marked by the following criteria, adjusting for the overall quality.

### P-level expectation
- Showing good effort through completed tasks.
- Applying deep learning theory to design suitable deep learning solutions for the tasks.
- Justify design choices.
- Identifies potential dataset biases and ethical concerns.

### C-level expectation
- Showing attention to detail through a structured and detailed assignment report.

### D-level expectation
- Demonstrating creativity and resourcefulness in providing unique individual solutions.
- Critically evaluating and reflecting on the pros and cons of various design decisions.

### HD-level expectation
- Extending classroom learning to research and tackle previously unexplored theoretical questions or novel applications.
- Critically reflects on the broader implications of deep learning models, including ethical and safety considerations.

**(Warning: Highly similar solutions will be investigated for collusion.)**

---


## **Set 1 Build a Simple Neural Network (P-Level Tasks)**
*Objective: Build a basic neural network from scratch and set up a reproducible training pipeline.*

You will practice training a neural network for a regression or a classification task. Completing this step will provide you a good foundation for later assessment tasks in this unit.

1. Determine a machine learning problem (e.g., Forecasting Energy Consumption in a Building) that you want to solve with neural networks. Describe the problem, objectives, and potential ethical concerns, including dataset biases.

2. Select an appropriate dataset for your selected problem, justify its suitability.  Consider dataset characteristics such as size, complexity, and potential biases. Preprocess the dataset for training.

3. Define a simple fully connected neural network using PyTorch.

4. Implement the training pipeline. Train the model and evaluate its performance on test data.




In [ ]:
# !pip install numpy
# !pip install pandas
# !pip install matplotlib
# !pip install torch torchvision torchaudio
# !pip install tensorboard
# !pip install google-cloud-storage

In [ ]:
# !nvidia-smi

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, r2_score
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import time

In [ ]:
!gcloud storage ls gs://sit744-bucket/DL/ass

gs://sit744-bucket/DL/ass/
gs://sit744-bucket/DL/ass/task1/


In [ ]:
# Define the GCS path to your CSV file
gcs_csv_path = 'gs://sit744-bucket/DL/ass/task1/EV_Energy_Consumption_Dataset.csv'
df = pd.read_csv(gcs_csv_path)
print(df.head())
print(f"\nDataFrame shape: {df.shape}")

   Vehicle_ID            Timestamp   Speed_kmh  Acceleration_ms2  \
0        1102  2024-01-01 00:00:00  111.507366         -2.773816   
1        1435  2024-01-01 00:01:00   48.612323         -0.796982   
2        1860  2024-01-01 00:02:00  108.733320          0.253800   
3        1270  2024-01-01 00:03:00   38.579484         -2.111395   
4        1106  2024-01-01 00:04:00   57.172438          1.477883   

   Battery_State_%  Battery_Voltage_V  Battery_Temperature_C  Driving_Mode  \
0        30.415148         378.091525              25.314786             2   
1        97.385534         392.718377              18.240755             1   
2        84.912600         398.993495              44.449145             1   
3        28.777904         358.128273              28.980155             1   
4        29.740160         310.888162              33.184551             2   

   Road_Type  Traffic_Condition   Slope_%  Weather_Condition  Temperature_C  \
0          1                  1  6.879446  

In [ ]:
# df.describe()

In [ ]:
X = df.loc[:, df.columns.difference(['Vehicle_ID', 'Timestamp', 'Energy_Consumption_kWh'])]
y = df.loc[:, 'Energy_Consumption_kWh']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(4000, 16) (1000, 16) (4000,) (1000,)


In [ ]:
# convert to tensor for gpu accelation and pytorch
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1) # reshare [4000,] to [4000,1]
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [ ]:
# Prepare datasets and dataloaders
# TensorDataset wraps input (X_train_tensor) and target/output (y_train_tensor) tensors together
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# handling mini-batches for training align with tensor core will be better
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
def mean_absolute_error(preds, targets):
  return torch.mean(torch.abs(preds - targets))

In [ ]:
# Define the model
class EVEnergyModel(nn.Module):
    def __init__(self, input_dim):
        # input(16) > 32 > 16 > 16 > 1
        super(EVEnergyModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available and enabled for PyTorch.")
else:
    device = torch.device("cpu")
    print("GPU is not available. Using CPU instead.")

GPU is not available. Using CPU instead.


In [ ]:
model = EVEnergyModel(input_dim=X_train.shape[1])
model.to(device)

# Initialize TensorBoard writer
writer1 = SummaryWriter(log_dir="runs/experiment_1")

# create a sample match with input batch size=1 for Tensorboard
sample_input = torch.randn(1, X_train.shape[1]).to(device)

# Log computational graph to TensorBoard
writer1.add_graph(model, sample_input)
writer1.flush()

In [ ]:
# loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.02)

epochs = 500
for epoch in range(epochs):
  # train mode (work Dropout or BatchNorm), pair with model.eval()
  model.train()
  # Keeps track of total loss for the epoch
  # accumulate for each mini-batch (average loss among batch and compare with each epoch)
  running_loss = 0.0
  running_mae = 0.0
  # mini-batch Traini
  for inputs, targets in train_loader:
      inputs, targets = inputs.to(device), targets.to(device)

      optimizer.zero_grad() # Clears out previous gradients to avoid accumulation.
      outputs = model(inputs) # Makes predictions
      loss = criterion(outputs, targets) # Computes how far off the predictions i.e. Loss
      loss.backward() # Calculates gradients via backpropagation
      optimizer.step() # Updates the model weights based on gradients and learning rate.

      running_loss += loss.item() * inputs.size(0)
      running_mae += mean_absolute_error(outputs, targets).item() * inputs.size(0)

  epoch_loss = running_loss / len(train_dataset)
  epoch_mae = running_mae / len(train_dataset)
  writer1.add_scalar("Loss/train", epoch_loss, epoch)
  writer1.add_scalar("MAE/train", epoch_mae, epoch)
print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}, MAE: {epoch_mae:.4f}")

Epoch 500/500, Loss: 0.3546, MAE: 0.4746


In [ ]:
# # save model’s weights
# # save weights only, it needs to provide architecture before load the model
# torch.save(model.state_dict(), "evmodel.pth")
# print("Model saved to evmodel.pth")

In [ ]:
# # Construct same architecture as saved model
# input_dim = X_train.shape[1]  # input_dim need to be same dim
# model = EVEnergyModel(input_dim)

# # load saved model
# model.load_state_dict(torch.load("evmodel.pth"))

# # move to gpu if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# print("Model loaded and ready!")

# # continue to evaluate
# # model.eval()

In [ ]:
model.eval()
test_loss = 0.0
total_mae = 0.0
all_predictions = []
all_targets = []

# close backpropagation
with torch.no_grad():
  for inputs, targets in test_loader:
    inputs, targets = inputs.to(device), targets.to(device)
    predictions = model(inputs)
    # calculate loss
    loss = criterion(predictions, targets)
    test_loss += loss.item() * inputs.size(0)

    # calculate MAE
    mae = torch.abs(predictions - targets).sum()
    total_mae += mae.item()
    # move tensor to cpu, as numpy (to calculate R2) work on cpu only
    all_predictions.append(predictions.cpu())
    all_targets.append(targets.cpu())

test_loss /= len(test_dataset)
mean_mae = total_mae / len(test_dataset)

# calculate R2
all_predictions = torch.cat(all_predictions).numpy()
all_targets = torch.cat(all_targets).numpy()
r2 = r2_score(all_targets, all_predictions)

print(f"Test Loss (MSE): {test_loss:.4f}")
print(f"Test MAE: {mean_mae:.4f}")
print(f"Test R² score: {r2:.4f}")
writer1.close()

Test Loss (MSE): 0.5651
Test MAE: 0.6196
Test R² score: 0.8835


In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs --port=6005

Reusing TensorBoard on port 6005 (pid 1641), started 2:17:21 ago. (Use '!kill 1641' to kill it.)

<IPython.core.display.Javascript object>

## **Set 2 Improve Model Performance (C-Level Tasks)**

*Objective: Modify and refine the model to improve performance while maintaining computational efficiency.*

1. Use TensorBoard to log loss and accuracy.
2. Modify the network architecture to improve performance.
3. Apply machine learning principles to adjust training configurations (e.g., learning rate, batch size).
4. Analyse model training using TensorBoard and compare different runs.
5. Justify design choices based on practical outcomes.



In [ ]:
class EVEnergyModelImproved(nn.Module):
    def __init__(self, input_dim):
        super(EVEnergyModelImproved, self).__init__()
        self.fc1 = nn.Linear(input_dim, 48)
        self.fc2 = nn.Linear(48, 24)
        self.fc3 = nn.Linear(24, 12)
        self.fc4 = nn.Linear(12, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available and enabled for PyTorch.")
else:
    device = torch.device("cpu")
    print("GPU is not available. Using CPU instead.")

GPU is not available. Using CPU instead.


In [ ]:
model = EVEnergyModelImproved(input_dim=X_train.shape[1])
model.to(device)

# Initialize TensorBoard writer
writer2 = SummaryWriter(log_dir="runs/experiment_2")

# create a sample match with input batch size=1 for Tensorboard
sample_input = torch.randn(1, X_train.shape[1]).to(device)

# Log computational graph to TensorBoard
writer2.add_graph(model, sample_input)
writer2.flush()

In [ ]:
# loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 500
for epoch in range(epochs):
  # train mode (work Dropout or BatchNorm), pair with model.eval()
  model.train()
  # Keeps track of total loss for the epoch
  # accumulate for each mini-batch (average loss among batch and compare with each epoch)
  running_loss = 0.0
  running_mae = 0.0
  # mini-batch Training
  for inputs, targets in train_loader:
      inputs, targets = inputs.to(device), targets.to(device)

      optimizer.zero_grad() # Clears out previous gradients to avoid accumulation.
      outputs = model(inputs) # Makes predictions
      loss = criterion(outputs, targets) # Computes how far off the predictions i.e. Loss
      loss.backward() # Calculates gradients via backpropagation
      optimizer.step() # Updates the model weights based on gradients and learning rate.

      running_loss += loss.item() * inputs.size(0)
      running_mae += mean_absolute_error(outputs, targets).item() * inputs.size(0)

  epoch_loss = running_loss / len(train_dataset)
  epoch_mae = running_mae / len(train_dataset)
  writer1.add_scalar("Loss/train", epoch_loss, epoch)
  writer1.add_scalar("MAE/train", epoch_mae, epoch)
print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}, MAE: {epoch_mae:.4f}")

Epoch 500/500, Loss: 0.2926, MAE: 0.4302


In [ ]:
model.eval()
test_loss = 0.0
total_mae = 0.0
all_predictions = []
all_targets = []

# close backpropagation
with torch.no_grad():
  for inputs, targets in test_loader:
    inputs, targets = inputs.to(device), targets.to(device)
    predictions = model(inputs)
    # calculate loss
    loss = criterion(predictions, targets)
    test_loss += loss.item() * inputs.size(0)

    # calculate MAE
    mae = torch.abs(predictions - targets).sum()
    total_mae += mae.item()
    # move tensor to cpu, as numpy (to calculate R2) work on cpu only
    all_predictions.append(predictions.cpu())
    all_targets.append(targets.cpu())

test_loss /= len(test_dataset)
mean_mae = total_mae / len(test_dataset)

# calculate R2
all_predictions = torch.cat(all_predictions).numpy()
all_targets = torch.cat(all_targets).numpy()
r2 = r2_score(all_targets, all_predictions)

print(f"Test Loss (MSE): {test_loss:.4f}")
print(f"Test MAE: {mean_mae:.4f}")
print(f"Test R² score: {r2:.4f}")
writer1.close()

Test Loss (MSE): 0.3111
Test MAE: 0.4480
Test R² score: 0.9359


In [ ]:
%reload_ext tensorboard
%tensorboard --logdir runs --port=6005

Reusing TensorBoard on port 6005 (pid 1641), started 2:19:45 ago. (Use '!kill 1641' to kill it.)

<IPython.core.display.Javascript object>

## **Set 3 Ethical Analysis and Model Evaluation (D-Level Tasks)**

*Objective: Investigate the ethical implications of deep learning applications through both critical reflection and hands-on model evaluation.*

1. Analyze dataset biases programmatically by visualising class distributions and measuring imbalances. Assess how these biases impact model performance.
2. Experiment with mitigation techniques, such as rebalancing the dataset, modifying loss functions, or applying bias correction strategies. Compare results.
3. Evaluate model performance critically by testing it on diverse subsets of data and discussing the strengths and limitations based on empirical findings.

## **Set 4 Reproducing and Analyzing "Grokking" in Neural Networks (HD-Level Tasks)**

*Objective:
Develop research and analytical skills by reproducing key findings from a research paper, identifying inconsistencies in experimental results, and critically evaluating the generalization behaviors of deep learning models.*

1. Reproduce key experiments from the paper ["OMNIGROK: GROKKING BEYOND ALGORITHMIC DATA"](https://arxiv.org/pdf/2210.01117).

2. Modify the experiments and identify experiment settings that may not be consistent with the authors' conclusions.

3. Provide a critical analysis discussing the potential causes of inconsistencies and evaluate the paper’s claims.

Deliverables:

- A Jupyter notebook containing:

  - Code for reproducing experiments (only on two datasets).

  - Modified experimental setups and results.

  - Visualizations and structured data analysis.

- A written report (PDF, max 2 pages) including:

  - Summary of reproduction steps.

  - Description of inconsistencies and contradictory findings.

  - A critical evaluation of possible causes.

- Short video/screencast (max 5 minutes) explaining key findings and insights.


---

### End of Assignment 1

---